# DocLens + Confidence-gated Abstention (CDC) on Kaggle

This notebook
1. clones **DocLens** (https://github.com/dwzhu-pku/DocLens) and the **CDC extension** (`doclens_cdc/`),
2. builds a small MMLongBench-Doc subset and **degraded copies** of it (blur, noise, JPEG, low resolution, mixed),
3. parses every copy with **MinerU** on the Kaggle GPU (DocLens' Element Localizer),
4. runs **original DocLens** and **DocLens + Confidence and Degradation Check**,
5. reports accuracy, over-refinement, abstention rate and token cost per degradation.

**Before running**
* *Settings → Accelerator*: **GPU T4 x2** (or P100). *Settings → Internet*: **On**.
* *Add-ons → Secrets*: add `GEMINI_API_KEY` (a Google AI Studio key: https://aistudio.google.com/apikey) and attach it to this notebook.
  A free key works for the smoke test; the full experiment makes several hundred multimodal calls, so a billing-enabled key is strongly recommended.
* If the repository holding `doclens_cdc` is private, also add a `GITHUB_TOKEN` secret (read access).

In [ ]:
# ---- Experiment settings (edit here) ----
WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"

DATASET = "MMLongBenchDoc"
MAX_DOCS, MAX_PAGES, MAX_QUESTIONS = 8, 30, 60      # subset size (MinerU parses MAX_DOCS*<=MAX_PAGES pages per variant)
VARIANTS = ["clean", "blur_s2", "noise_s2", "jpeg_s3", "lowres_s2", "mixed_s2"]

MODEL = "gemini-2.5-flash"            # Answer Sampler + Adjudicator (DocLens paper's best: gemini-2.5-pro)
NAVIGATOR_MODEL = "gemini-2.5-flash-lite"   # DocLens' Page Navigator default
EVAL_MODEL = "gemini-2.5-flash"       # DocLens' answer extractor for scoring
RPM = 10                              # requests/minute cap (raise for paid keys)
MAX_CONCURRENT = 4

## 1. Get the code

In [ ]:
import os, subprocess, sys, shutil
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dwzhu-pku/DocLens.git"], check=True)

if not os.path.exists("doclens_cdc"):
    url = CDC_REPO
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = CDC_REPO.replace("https://", f"https://{token}@")
    except Exception:
        pass  # public repository: no token needed
    subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, url, "_cdc_repo"], check=True)
    shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
    shutil.rmtree("_cdc_repo")

DOCLENS = f"{WORK}/DocLens"
CDC = f"{WORK}/doclens_cdc"
VROOT = f"{WORK}/variants"
RESULTS = f"{WORK}/results"
os.makedirs(RESULTS, exist_ok=True)
print(os.listdir(CDC))

## 2. Install dependencies
MinerU is the document parser DocLens uses for element localization. This takes a few minutes.

In [ ]:
!pip install -q -U "mineru[core]" 2>&1 | tail -n 3
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 3
!mineru --version

## 3. Gemini API key
DocLens uses Vertex AI by default; the CDC runner swaps in an API-key client, so no Google Cloud project is needed.

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")

from google import genai
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print(client.models.generate_content(model=MODEL, contents="Reply with the word OK.").text)

## 4. Download MMLongBench-Doc and build a subset

In [ ]:
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data

In [ ]:
# DocLens expects data/<dataset>/samples.json and data/<dataset>/documents/*.pdf.
# Adjust these two paths if the listing above shows a different layout.
import glob
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"
assert os.path.exists(SAMPLES), glob.glob(f"{DOCLENS}/data/{DATASET}/**/*.json", recursive=True)[:10]
print(len(glob.glob(f"{DOCUMENTS}/*.pdf")), "PDFs")

In [ ]:
!python {CDC}/scripts/prepare_subset.py --samples {SAMPLES} --documents {DOCUMENTS} --out_root {VROOT} \
    --dataset_name {DATASET} --max_docs {MAX_DOCS} --max_pages {MAX_PAGES} --max_questions {MAX_QUESTIONS}
SUBSET = f"{VROOT}/samples_subset.json"

## 5. Create degraded copies
Each variant is a full DocLens data root: `variants/<variant>/data/<dataset>/documents/...`.

In [ ]:
degraded = " ".join(v for v in VARIANTS if v != "clean")
!python {CDC}/scripts/make_variants.py --out_root {VROOT} --dataset_name {DATASET} --variants {degraded}

## 6. Parse every variant with MinerU (GPU)
Degraded pages are re-parsed, so the OCR text the agents see is degraded too — as in a real scanning pipeline. The script is resumable.

In [ ]:
for v in VARIANTS:
    print("=" * 20, v)
    !python {CDC}/scripts/parse_mineru.py --root {VROOT}/{v} --dataset_name {DATASET}

## 7. (Optional) Calibrate the degradation detector on these documents
Fits the quality thresholds to your documents and reports how well degradation is detected/classified. For a clean evaluation, calibrate on documents you do not evaluate on.

In [ ]:
CALIB = f"{RESULTS}/calibration.json"
cal_variants = " ".join(v for v in VARIANTS if v != "clean")
!python {CDC}/scripts/calibrate_cdc.py --variants_root {VROOT} --dataset_name {DATASET} --variants {cal_variants} --out {CALIB}
USE_CALIBRATION = False   # set True to use the fitted thresholds in the runs below

## 8. Smoke test (2 questions)

In [ ]:
calib_arg = f"--calibration {CALIB}" if USE_CALIBRATION else ""
common = (f"--doclens_root {DOCLENS} --dataset_name {DATASET} --samples {SUBSET} --model_name {MODEL} "
          f"--navigator_model {NAVIGATOR_MODEL} --eval_model {EVAL_MODEL} --rpm {RPM} "
          f"--max_concurrent {MAX_CONCURRENT} {calib_arg}")
!python {CDC}/run_cdc.py {common} --data_root {VROOT}/clean --mode cdc --limit 2 --output {RESULTS}/smoke.jsonl

In [ ]:
import json
for line in open(f"{RESULTS}/smoke.jsonl"):
    r = json.loads(line)
    print(r["question"][:80], "| gold:", r.get("answer"), "| pred:", r.get("final_prediction"),
          "| decision:", r.get("cdc_decision"), f"(conf {r.get('cdc_confidence', 0):.2f})", "| error:", "cdc_error" in r)

## 9. Main experiment
`--mode analyze` runs the Lens Module once, computes the CDC report, and then runs **all three** reasoning paths
(abstain / correct / original DocLens) for every question. This costs ~3x a single run but lets us evaluate *any*
threshold offline, measure over-refinement exactly, and compare against original DocLens on identical Lens outputs.
Re-running a cell resumes where it stopped.

In [ ]:
for v in VARIANTS:
    print("=" * 20, v)
    !python {CDC}/run_cdc.py {common} --data_root {VROOT}/{v} --variant {v} --mode analyze --output {RESULTS}/{v}_analyze.jsonl

## 10. Results

In [ ]:
!python {CDC}/scripts/analyze_results.py {RESULTS}/*_analyze.jsonl --out_dir {RESULTS}/report --dev_frac 0.3

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{RESULTS}/report/report.md").read()))
if os.path.exists(f"{RESULTS}/report/threshold_sweep.png"):
    display(Image(f"{RESULTS}/report/threshold_sweep.png"))

## 11. Deployment-style run (optional)
`--mode cdc` runs only the path the gate picks (this is the configuration whose cost you would report).
Use the threshold selected in the report above.

In [ ]:
TAU = 0.70   # replace with the tau selected on the dev split
for v in VARIANTS:
    !python {CDC}/run_cdc.py {common} --data_root {VROOT}/{v} --variant {v} --mode cdc --tau_case {TAU} --output {RESULTS}/{v}_cdc.jsonl

Everything under `/kaggle/working/results` is kept when you *Save Version*. Download `results/report/` for the paper tables.